EXPLORACIÓN DEL DATASET NBA

QUÉ ES: Este notebook es para explorar el dataset, me va a ayudar a leer los ficheros originales y generarme un informe de perfil. Para cada uno de los cuatro ficheros va a recoger su tamaño, las filas duplicadas y por columna, el tipo de dato, el porcentaje de nulos y los valores distintos. 

He tomado este paso por tres motivos:

    1. El fichero principal supera el millon de filas, lo que hace imposible revisarlo a mano o en una hoja de cálculo sin perder datos. 

    2. Permite que cada cifra de la ficha tenga una fuente verificable. 

    3. Siguiendo las lecciones 02 y 03, documentar el estado de los datos el día de la descarga y cuantificar sus problemas de calidad desde el principio sirve como punto de partida para la fase de limpieza y para detectar cambios si la fuente se actualiza. 

DE DÓNDE SALEN LOS DATOS: NBA Dataset: Box Scores and STats (1947 - Today), de EUIN A.Moore, en Kaggle, licencia CC0. 

ACLARACIÓN: el notebook solo lee los ficheros originales y no los modifica. 


In [1]:
##Preparacion
from pathlib import Path #trae la herramienta para manejar rutas de carpetas
import pandas as pd #trae pandas

RAW =Path("..")/"data"/"raw" #guarda donde estan los CSV 
                             #.. es "sube una carpeta", desde notebooks/a project/.
pd.set_option("display.max_columns", None)

In [2]:
#cargar Players.csv y mostrar sus primeras filas

players = pd.read_csv(RAW/"Players.csv")
players.head()

,personId,firstName,lastName,birthDate,school,country,heightInches,bodyWeightLbs,jersey,guard,forward,center,dleagueFlag,nbaFlag,gamesPlayedFlag,draftYear,draftRound,draftNumber,fromYear,toYear
0,196294534,Alberto,Abalde,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,76001,Alaa,Abdelnaby,1968-06-24,Duke,USA,82.0,240.0,30,0,1,0,0.0,1.0,1.0,1990.0,1.0,25.0,1990.0,1994.0
2,76002,Zaid,Abdul-Aziz,1946-04-07,Iowa State,USA,81.0,235.0,54,0,0,1,0.0,1.0,1.0,1968.0,1.0,5.0,1968.0,1977.0
3,76003,Kareem,Abdul-Jabbar,1947-04-16,UCLA,USA,86.0,225.0,33,0,0,1,0.0,1.0,1.0,1969.0,1.0,1.0,1969.0,1988.0
4,51,Mahmoud,Abdul-Rauf,1969-03-09,Louisiana State,USA,73.0,162.0,1,1,0,0,0.0,1.0,1.0,1990.0,1.0,3.0,1990.0,2000.0


## ¿QUÉ CONTIENE LA TABLA?
```
personID: identificador
Nombre y apellidos (campos separados)
Fecha de nacimiento
Universidad
País
Altura (pero en nomenclatura rara)
Peso
Numero camiseta
Posición
dleagFlag y nbaFlag
Partidos jugados en Flag
Año de Draft
Draft round
Draft number
De que año a que año
```

El primer hallazco de calidad: ALBERTO(0) no tiene fecha de nacimiento con la cual quiero calcular la edad para asociar que cuanto mayor es el jugador hay posibilidad de más fatiga.

Otro dato que me llama la atención es que dleagueFlag y nbaFlag sus datos estan en decimal .0, 0.0 y 1.0, creo que quieren decir 0 y 1... hay que corregir e indagar más adelante en ello. 

In [3]:
#cargar los otros tres csv y mostrar sus primeras filas
#cargar Games.cvs
games = pd.read_csv(RAW/"Games.csv", low_memory=False) #low_memory=False evita un warning
games.head()

,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,gameSubtype,gameLabel,gameSubLabel,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate
0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,90,94,1610612752,Playoffs,NaN,NBA Finals,Game 5,Game 5,18984.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, Scott Foster, Mitchell Ervin, Ty...",2026-06-13 20:30:00
1,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,107,106,1610612752,Playoffs,NaN,NBA Finals,Game 4,Game 4,19812.0,30,Madison Square Garden,New York,NY,"Courtney Kirkland, Zach Zarba, James Williams,...",2026-06-10 20:30:00
2,42500403,2026-06-08 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,111,115,1610612759,Playoffs,NaN,NBA Finals,Game 3,Game 3,19812.0,30,Madison Square Garden,New York,NY,"Marc Davis, John Goble, Curtis Blair, Nick Buc...",2026-06-08 20:30:00
3,42500402,2026-06-05 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,104,105,1610612752,Playoffs,NaN,NBA Finals,Game 2,Game 2,19014.0,1000118,Frost Bank Center,San Antonio,TX,"Tony Brothers, Josh Tiven, Mitchell Ervin, Tyl...",2026-06-05 20:30:00
4,42500401,2026-06-03 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,95,105,1610612752,Playoffs,NaN,NBA Finals,Game 1,Game 1,18835.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, JB DeRosa, Scott Foster, Sean Wr...",2026-06-03 20:30:00


## ¿Qué hay en la tabla?

```
gameId                gameDateTimeEst       hometeamCity
hometeamName          hometeamId            awayteamCity
awayteamName          awayteamId            homeScore
awayScore             winner                gameType
gameSubtype           gameLabel             gameSubLabel
seriesGameNumber      attendance            arenaId
arenaName             arenaCity             arenaState
officials             gameDate
```


In [4]:
#cargar TeamStatistics.csv
team_stats = pd.read_csv(RAW/"TeamStatistics.csv")
team_stats.head()

,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,teamScore,opponentScore,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,numMinutes,q1Points,q2Points,q3Points,q4Points,benchPoints,biggestLead,biggestScoringRun,leadChanges,pointsFastBreak,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId,gameType,gameLabel,gameSubLabel,seriesGameNumber,seed,reboundsTeam,turnoversTeam,ot1Points,ot2Points,otAllPoints,gameDate
0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,1,0.0,90,94,18.0,7.0,6.0,86.0,33.0,0.384,37.0,12.0,0.324,19.0,12.0,0.632,33.0,14.0,47.0,22.0,12.0,-4.0,240.0,23.0,19.0,30.0,18.0,32.0,16.0,9.0,5.0,13.0,21.0,38.0,17.0,5.0,0.0,1.0,4.0,NaN,Playoffs,NBA Finals,Game 5,5.0,2.0,12.0,1.0,NaN,NaN,NaN,2026-06-13 20:30:00
1,42500405,2026-06-13 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,0,1.0,94,90,14.0,4.0,8.0,87.0,31.0,0.356,37.0,12.0,0.324,28.0,20.0,0.714,35.0,13.0,48.0,21.0,10.0,4.0,240.0,13.0,24.0,28.0,29.0,9.0,4.0,10.0,5.0,13.0,18.0,30.0,14.0,5.0,0.0,4.0,1.0,NaN,Playoffs,NBA Finals,Game 5,5.0,3.0,18.0,4.0,NaN,NaN,NaN,2026-06-13 20:30:00
2,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,1,1.0,107,106,23.0,4.0,6.0,78.0,36.0,0.462,32.0,15.0,0.469,28.0,20.0,0.714,31.0,8.0,39.0,21.0,13.0,1.0,240.0,22.0,27.0,26.0,32.0,12.0,1.0,13.0,3.0,13.0,11.0,34.0,14.0,2.0,0.0,3.0,1.0,NaN,Playoffs,NBA Finals,Game 4,4.0,3.0,14.0,2.0,NaN,NaN,NaN,2026-06-10 20:30:00
3,42500404,2026-06-10 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,0,0.0,106,107,24.0,4.0,10.0,86.0,36.0,0.419,43.0,17.0,0.395,20.0,17.0,0.850,30.0,12.0,42.0,21.0,11.0,-1.0,240.0,41.0,35.0,14.0,16.0,28.0,29.0,10.0,3.0,12.0,25.0,28.0,10.0,2.0,0.0,1.0,3.0,NaN,Playoffs,NBA Finals,Game 4,4.0,2.0,8.0,1.0,NaN,NaN,NaN,2026-06-10 20:30:00
4,42500403,2026-06-08 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,0,1.0,115,111,28.0,7.0,7.0,84.0,39.0,0.464,34.0,12.0,0.353,32.0,25.0,0.781,31.0,6.0,37.0,21.0,8.0,4.0,240.0,33.0,24.0,35.0,23.0,25.0,12.0,7.0,6.0,5.0,21.0,44.0,10.0,4.0,0.0,1.0,2.0,NaN,Playoffs,NBA Finals,Game 3,3.0,2.0,11.0,0.0,NaN,NaN,NaN,2026-06-08 20:30:00


## ¿Qué hay en la tabla?

```
gameId                    gameDateTimeEst           teamCity                  teamName
teamId                    opponentTeamCity          opponentTeamName          opponentTeamId
home                      win                       teamScore                 opponentScore
assists                   blocks                    steals                    fieldGoalsAttempted
fieldGoalsMade            fieldGoalsPercentage      threePointersAttempted    threePointersMade
threePointersPercentage   freeThrowsAttempted       freeThrowsMade            freeThrowsPercentage
reboundsDefensive         reboundsOffensive         reboundsTotal             foulsPersonal
turnovers                 plusMinusPoints           numMinutes                q1Points
q2Points                  q3Points                  q4Points                  benchPoints
biggestLead               biggestScoringRun         leadChanges               pointsFastBreak
pointsFromTurnovers       pointsInThePaint          pointsSecondChance        timesTied
timeoutsRemaining         seasonWins                seasonLosses              coachId
gameType                  gameLabel                 gameSubLabel              seriesGameNumber
seed                      reboundsTeam              turnoversTeam             ot1Points
ot2Points                 otAllPoints               gameDate
```

Tengo **seasonWins** y **seasonLosses**, con ellas puedo medir el nivel del rival -> descansar contra rivales débiles. 

In [5]:
#cargar PlayerStatistics.csv
player_stats = pd.read_csv(RAW/"PlayerStatistics.csv", low_memory=False) #low_memory=False evita un warning
player_stats.head()

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,gameLabel,gameSubLabel,seriesGameNumber,win,home,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate
0,Josh,Hart,1628404.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,39.166666666666664,13.0,2.0,0.0,0.0,11.0,4.0,0.364,6.0,3.0,0.5,3.0,2.0,0.667,8.0,3.0,11.0,3.0,1.0,15.0,1.610613e+09,1.610613e+09,NaN,F,2026-06-13 20:30:00
1,OG,Anunoby,1628384.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,32.78333333333333,11.0,0.0,1.0,3.0,11.0,3.0,0.273,5.0,1.0,0.2,6.0,4.0,0.667,8.0,0.0,8.0,5.0,1.0,2.0,1.610613e+09,1.610613e+09,NaN,F,2026-06-13 20:30:00
2,Jeremy,Sochan,1631110.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,1.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-06-13 20:30:00
3,Jordan,McLaughlin,1629162.0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,New York,Knicks,Playoffs,NBA Finals,Game 5,5,0.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,1.610613e+09,1.610613e+09,DNP - Coach's Decision,NaN,2026-06-13 20:30:00
4,Jordan,Clarkson,203903.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,6.116666666666666,2.0,0.0,0.0,0.0,5.0,1.0,0.200,2.0,0.0,0.0,0.0,0.0,0.000,0.0,1.0,1.0,0.0,0.0,3.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-06-13 20:30:00


## ¿Qué hay en la tabla?

```
firstName                 lastName                  personId
gameId                    gameDateTimeEst           playerteamCity
playerteamName            opponentteamCity          opponentteamName
gameType                  gameLabel                 gameSubLabel
seriesGameNumber          win                       home
numMinutes                points                    assists
blocks                    steals                    fieldGoalsAttempted
fieldGoalsMade            fieldGoalsPercentage      threePointersAttempted
threePointersMade         threePointersPercentage   freeThrowsAttempted
freeThrowsMade            freeThrowsPercentage      reboundsDefensive
reboundsOffensive         reboundsTotal             foulsPersonal
turnovers                 plusMinusPoints           playerteamId
opponentteamId            comment                   startingPosition
gameDate
```

Esta tabla tiene mucha información que va a ser esencial para mi sección de carga:

- **Carga** (minutos jugados) → `numMinutes`
- **Fecha** (calcular descansos y back-to-backs) → `gameDateTimeEst` / `gameDate`
- **Rendimiento** → `points`, `plusMinusPoints`, los tiros
- **Contexto** → `home`, `win`, `opponentteamName`, `gameType`
- **Jugador** → `personId`

Por otro lado, **comment** creo que va a ser oro par mi proyecto: En la 4ª fila pone “DNP - Coach’s Decision”. DNP significa Did Not Play: el jugador no jugó porque el entrenador lo decidió.Si esta columna recogiera otros motivos, como lesión, tendría una aproximación a lesiones y descansos. 



## TAMAÑO, TIPOS DE DATOS Y NULOS

En este bloque mido cada tabla:
- cuantas filas y columnas tiene, 
- qué tipo de dato hay en cada columna, 
- cuántos valores faltan. 


### Tamaño de cada tabla


In [6]:
print("players:", players.shape) 
print("games:", games.shape) #con shape se puede ver cuantas filas y columnas tiene el dataframe
print("team_stats:", team_stats.shape)
print("player_stats:", player_stats.shape)
#resultado tapla_name : (filas, columnas)

players: (6692, 20)
games: (73279, 23)
team_stats: (146560, 59)
player_stats: (1669922, 40)


### Tipos de dato

Aqui voy a revisar qué tipo ha asignado pandas a cada columna de las dos tablas que más voy a usar: `player_stats` y `players`.

In [7]:
player_stats.dtypes

firstName                      str
lastName                       str
personId                   float64
gameId                       int64
gameDateTimeEst                str
playerteamCity                 str
playerteamName                 str
opponentteamCity               str
opponentteamName               str
gameType                       str
gameLabel                      str
gameSubLabel                   str
seriesGameNumber               str
win                        float64
home                       float64
numMinutes                     str
points                     float64
assists                    float64
blocks                     float64
steals                     float64
fieldGoalsAttempted        float64
fieldGoalsMade             float64
fieldGoalsPercentage       float64
threePointersAttempted     float64
threePointersMade          float64
threePointersPercentage    float64
freeThrowsAttempted        float64
freeThrowsMade             float64
freeThrowsPercentage

**Observaciones**: numMinutes es texto, y es una de mis variables principales sobre las que quiero trabajar. Algo pasa... voy a sarcar 20 filas al azar a ver que es lo que esta pasando.

In [8]:
player_stats["numMinutes"].sample(20)

310589      25.0
573838      28.0
1631410      0.0
435521      36.0
1422177     12.0
11128      29.24
56648      18.24
656626       NaN
1230616     44.0
434266      31.0
165607      35.0
257079       3.0
179862       1.0
325172      23.0
178128      16.0
1455747     31.0
135252      15.0
779606       NaN
1059372     40.0
877984      30.0
Name: numMinutes, dtype: str

La muestra parece normal, casi todo son números y dos NaN. Aun así me sigue molestando que dtype sea str. 

Lo que puedo ver **ahora mismo**:
- Los minutos estan redondeados a enteros, lo que me quita precisión y debo comentarlo cuando llegue a mis conclusiones. 
- Hay 0.0 y NaN que no es lo mismo. 0.0 significaría que el jugados estaba en el partido pero no ha jugado nada y NaN es que el dato falta. 

Debería encontrar los valores raros que hacen que los minutos se declaren como str. 


In [9]:
# Intento convertir numMinutes a número; lo que no se pueda convertir se queda vacío (NaN)
minutos_num = pd.to_numeric(player_stats["numMinutes"], errors="coerce")

# Valores que NO estaban vacíos en el original pero SÍ quedan vacíos al convertir: esos son los raros
raros = player_stats.loc[minutos_num.isna() & player_stats["numMinutes"].notna(), "numMinutes"]

print("Filas con valores no numéricos:", len(raros))
raros.value_counts().head(20)

Filas con valores no numéricos: 2097


numMinutes
12:00    28
22:29     7
23:46     6
15:51     6
16:08     6
14:53     6
7:04      6
15:43     6
16:22     5
8:27      5
21:18     5
21:15     5
1:54      5
17:16     5
16:07     5
17:11     5
20:47     5
16:42     5
16:53     5
17:34     5
Name: count, dtype: int64

Ya tengo el culpable de que me salga el tipo str: tiene 2097 filas en formato minutos:segundos y el resto utiliza un número decimal. 

Al saber la causa, hay que resolverlo bien. Plantearme si descartar todas las filas ya que de 1,6 millones de filas son solo 0,1 % aproximadamente o si deberia convertir el 22:34 en 22,34. 



In [10]:
players.dtypes

personId             int64
firstName              str
lastName               str
birthDate              str
school                 str
country                str
heightInches       float64
bodyWeightLbs      float64
jersey                 str
guard                int64
forward              int64
center               int64
dleagueFlag        float64
nbaFlag            float64
gamesPlayedFlag    float64
draftYear          float64
draftRound         float64
draftNumber        float64
fromYear           float64
toYear             float64
dtype: object

Observaciones: 
- 'birthDate' es una cadena de código si quiero calcular la edad tendre que convertirla. 
- las unidades de 'hightInches' y 'bodyWeightLbs' estan en pulgadas y libras tendre que cambiarlas a cm y kg ya que estoy en España. 